In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),  # pyright: ignore[reportArgumentType]
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),  # pyright: ignore[reportArgumentType]
    extra_body={"enable_thinking": False},
)

In [ ]:
from typing import Literal
from langchain.tools import tool


@tool
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers."""
    print("🧮 Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage


agent = create_agent(
    model=model,
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke({"messages": [HumanMessage("3.1125 乘 4.1234")]})

print(result["messages"][-1].content)

🧮 Invoking calculator tool
3.1125 乘以 4.1234 的结果是 12.8340825。


In [6]:
result = agent.invoke({"messages": [HumanMessage("三乘四")]})

print(result["messages"][-1].content)

🧮 Invoking calculator tool
三乘四等于12。


In [7]:
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [8]:
agent = create_agent(
    model=model,
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke({"messages": [HumanMessage("3 乘 4.1234")]})

print(result["messages"][-1].content)

🧮  Invoking calculator tool
3 乘以 4.1234 的结果是 12.3702。


In [9]:
from pydantic import BaseModel, Field
from typing import Literal


class CalculatorInput(BaseModel):
    """Input for calculator."""

    a: float = Field(description="The first number")
    b: float = Field(description="The second number")
    operation: Literal["add", "subtract", "multiply", "divide"] = Field(
        ...,
        description="""The arithmetic operation to perform.
            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.""",
    )


@tool(
    "calculator",
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
    args_schema=CalculatorInput,
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [10]:
agent = create_agent(
    model=model,
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke({"messages": [HumanMessage("3 乘 4.1234")]})

print(result["messages"][-1].content)

🧮  Invoking calculator tool
3 乘以 4.1234 的结果是 12.3702。
